# Step 6 — Self-Supervised Pre-Training: `1D-ResNet-34 (Barlow Twins)` in Pure PyTorch (`716,686` Multi-Regime Spectra)

This HPC JupyterLab notebook runs our **Pure PyTorch port** of the thesis TensorFlow/Keras Barlow Twins model (Deliverable 3.2 Section 4.1: `1D-ResNet-34` without Global Average Pooling + 3-layer `[2048, 2048, 2048]` Projector + Cross-Correlation Redundancy Reduction Loss $\lambda_{\text{BT}} = 5\times 10^{-3}$) on the **entire unified `716,686`-spectrum corpus (`pretrain_cnn_all_1000.npz`: `FP + HW + FR`)**.

> **Before running on HPC JupyterLab**:
> 1. Run `pip install -e .` once in the HPC terminal inside the repository root.
> 2. Upload `data/02_ready/pretrain/CNN/pretrain_cnn_all_1000.npz` (`2.74 GB`) into `data/02_ready/pretrain/CNN/`.

## 1. HPC Environment & GPU Verification
Verify CUDA GPU availability, VRAM, and mixed-precision (`bf16`/`fp16`) support.

In [ ]:
from pathlib import Path
import torch, yaml, numpy as np, pandas as pd
from IPython.display import Image, display

ROOT = Path('..').resolve()
print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU device      :', torch.cuda.get_device_name(0))
    print('VRAM (GB)       :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

## 2. Load Pre-Training Configuration (`configs/train/barlow_twins_all_1000.yaml`)
Inspect backbone (`resnet34`, `pool_mode='flatten'`), projector (`[2048, 2048, 2048]`), physical Raman augmentations, and training schedule.

In [ ]:
CONFIG_PATH = ROOT / 'configs' / 'train' / 'barlow_twins_all_1000.yaml'
cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
print(yaml.safe_dump(cfg, sort_keys=False))

## 3. Inspect the Unified Multi-Regime Corpus (`716,686` Unique Spectra: `FP + HW + FR`)
Verify exact spectrum counts across all 10 constituent pre-training sources.

In [ ]:
npz_path = ROOT / cfg['data']['npz_path']
arc = np.load(npz_path, allow_pickle=True)
print(f"Loaded: {npz_path.name} | Shape: {arc['spectra'].shape}")
for name, cnt in zip(arc['source_names'], arc['source_counts']):
    print(f'  - {str(name):22s}: {int(cnt):7,d} spectra')

## 4. Verify Pure PyTorch `BarlowTwins1D` (`ResNet-34 + AvgPool1d(2) + Flatten`) & Preview Augmented Views
Matches `Barlow_Twins_Resnet_34.ipynb` Cell 13: inspect two correlated augmented views of representative spectra (`FP`, `HW`, `FR`) before launching pre-training.

In [ ]:
from spectrabreast.models import BarlowTwins1D
from spectrabreast.training import TorchRamanAugmentor, plot_barlow_twins_augmented_views

m_cfg, a_cfg = cfg['model'], cfg['augmentations']
bt_check = BarlowTwins1D(backbone=m_cfg['backbone'], pool_mode=m_cfg['pool_mode'], projector_dims=m_cfg['projector_dims'], lambda_bt=m_cfg['lambda_bt'])
aug_check = TorchRamanAugmentor(max_shift_pts=a_cfg['max_shift_pts'], blur_kernel_max=a_cfg['blur_kernel_max'], zero_prob=a_cfg['zero_prob'])
sample_idx = [0, 300000, 650000]
v1, v2 = aug_check(torch.from_numpy(arc['spectra'][sample_idx].astype(np.float32)))
out = bt_check(v1, v2)
print(f"Encoder params: {sum(p.numel() for p in bt_check.encoder.parameters()):,d} | Out dim: {bt_check.encoder.out_dim} | Init Loss: {out['loss'].item():.2f}")
plot_barlow_twins_augmented_views(v1.numpy(), v2.numpy(), [str(arc['source_names'][arc['source_id'][i]]) for i in sample_idx])

## 5. Launch Pure PyTorch `Barlow Twins` Pre-Training (Auto-Resume Enabled)
Runs `train_barlow_twins_from_config` with GPU-accelerated `TorchRamanAugmentor`, saving `best_resnet_encoder.pth`, `best_barlow_twins_full.pth`, `history.csv`, and cross-correlation snapshots.

In [ ]:
from spectrabreast.training import train_barlow_twins_from_config

results = train_barlow_twins_from_config(
    config_path=CONFIG_PATH,
    repo_root=ROOT,
    resume=True,
    max_epochs_override=None,  # Set e.g. 3 for a quick smoke test!
)
print(f"Finished! Best Val Barlow Twins Loss = {results['best_val_loss']:.4f}")
print('Best ResNet encoder checkpoint:', results['best_encoder_checkpoint'])

## 6. Visualize Barlow Twins Learning Curves (Total Loss, Invariance vs Redundancy, Diagonal Convergence)
Verify that `Mean Diag C_ii -> 1.0` while `Mean |Off-Diag C_ij| -> 0.0`.

In [ ]:
fig_dir = results['output_dir'] / 'figures'
display(Image(filename=str(fig_dir / 'barlow_twins_learning_curves.png')))

## 7. Visualize Cross-Correlation Matrix `C` & Collapse Audit (Epoch 1 vs Final Epoch)
Exact Deliverable 3.2 Fig. 2 empirical cross-correlation heatmap and per-dimension standard deviation check.

In [ ]:
cc_figs = sorted(fig_dir.glob('barlow_twins_crosscorr_epoch_*.png'))
print('Saved cross-correlation snapshots:', [p.name for p in cc_figs])
if cc_figs:
    display(Image(filename=str(cc_figs[0])))
    if len(cc_figs) > 1:
        display(Image(filename=str(cc_figs[-1])))